# 01. PWM первого и второго рода

В этом ноутбуке сравниваются две разные модели из текущего кода:

- `pwm_kind1(...)`: вход уже задан на частоте `f_clk`, сравнение с несущей идет на каждом такте.
- `pwm_kind2_latched(...)`: входные значения приходят из FIFO/data-rate потока, одно значение удерживается весь PWM-период.

Это принципиальное различие: второй род меняет временную интерпретацию, если скорость FIFO не согласована с частотой PWM-обновления.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE
elif (HERE / "tutorials" / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE / "tutorials"
else:
    raise RuntimeError("Run this notebook from the repository root or from the tutorials folder")

path_text = str(TUTORIAL_DIR)
if path_text not in sys.path:
    sys.path.insert(0, path_text)

import matplotlib.pyplot as plt
import numpy as np

from tutorial_helpers import (
    configure_plots,
    grouped_fifo_channel_waveforms,
    load_pwm_lab,
    plot_bitstream,
    plot_channel_stack,
    plot_moving_average_reconstruction,
    plot_pwm_carrier_output,
    plot_spectra,
    print_peak_table,
    pwm_kind2_channel_waveforms,
    show_grouped_mapping,
    time_us,
)

pl = load_pwm_lab()
configure_plots()

## Общие параметры

Сделаем одинаковую низкочастотную синусоиду, но для первого рода она уже дискретизирована на `f_clk`, а для второго рода это по одному значению на PWM-период.

In [ ]:
config = pl.PwmConfig(f_clk=60e6, f_pwm=1e6, resolution_bits=8)
f_signal = 60e3
n_periods = 96
n_clk = n_periods * config.period_samples

_, x_kind1 = pl.sine_samples(freq=f_signal, sample_rate=config.f_clk, n_samples=n_clk, amplitude=0.85)
_, x_fifo = pl.sine_samples(freq=f_signal, sample_rate=config.actual_f_pwm, n_samples=n_periods, amplitude=0.85)

print(config)
print(f"period_samples = {config.period_samples}")
print(f"actual_f_pwm  = {config.actual_f_pwm / 1e6:.3f} MHz")

## PWM первого рода

Входной сигнал меняется на каждом такте `f_clk`, поэтому его можно напрямую сравнивать с текущей фазой несущей.

In [ ]:
y_kind1 = pl.pwm_kind1(x_kind1, config)
carrier_kind1 = np.resize(pl.triangle_carrier(config.period_samples), n_clk)

plot_pwm_carrier_output(
    x_kind1,
    carrier_kind1,
    y_kind1,
    sample_rate=config.f_clk,
    max_points=18 * config.period_samples,
    input_scale=2**config.resolution_bits,
    title="PWM kind 1",
);

## PWM второго рода

Здесь одно FIFO-значение защелкивается на целый период. Для верхнего графика вход расширен через `repeat`, чтобы было видно удержание уровня.

In [ ]:
y_kind2 = pl.pwm_kind2_latched(x_fifo, config)
x_latched = np.repeat(x_fifo, config.period_samples)
carrier_kind2 = np.resize(pl.triangle_carrier(config.period_samples), y_kind2.size)

plot_pwm_carrier_output(
    x_latched,
    carrier_kind2,
    y_kind2,
    sample_rate=config.f_clk,
    max_points=18 * config.period_samples,
    input_scale=2**config.resolution_bits,
    title="PWM kind 2: latched FIFO sample",
);

## Среднее за PWM-период

Для корректной низкочастотной части среднее за период должно идти рядом с исходными FIFO-значениями.

In [ ]:
avg_kind1 = pl.moving_average_decimate(y_kind1, config.period_samples)
avg_kind2 = pl.moving_average_decimate(y_kind2, config.period_samples)
period_time_ms = np.arange(n_periods) / config.actual_f_pwm * 1e3

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.plot(period_time_ms, x_fifo, "o-", markersize=3, label="FIFO/input samples")
ax.plot(period_time_ms, avg_kind1[:n_periods], ".-", label="kind 1 period average")
ax.plot(period_time_ms, avg_kind2, "s--", markersize=3, label="kind 2 period average")
ax.set_title("Period averages")
ax.set_xlabel("time, ms")
ax.set_ylabel("normalized amplitude")
ax.legend(loc="upper right");

## Сравнение спектров

Оба PWM-потока существуют на `f_clk`, поэтому спектр считаем с `sample_rate=config.f_clk`.

In [ ]:
plot_spectra(
    {"kind 1": y_kind1, "kind 2": y_kind2},
    sample_rate=config.f_clk,
    f_max=8e6,
    f_scale=1e6,
    f_unit="MHz",
    title="PWM kind 1 versus kind 2",
);

print_peak_table(
    {"kind 1": y_kind1, "kind 2": y_kind2},
    sample_rate=config.f_clk,
    f_min=1.0,
    f_max=8e6,
    f_scale=1e3,
    f_unit="kHz",
)

## Проверка FIFO-rate идеи

Если данные приходят быстрее, чем один PWM-канал может читать, одиночный `kind2` канал фактически читает поток медленнее. В текущем коде для этого есть готовая проверка `simulate_fifo_parallel_idea()`.

In [ ]:
result = pl.simulate_fifo_parallel_idea(
    f_data=1e6,
    f_pwm=200e3,
    f_clk=100e6,
    f_signal=20e3,
    channels=5,
    n_data_samples=4096,
)

for name, (freq, amp) in result["peaks_hz"].items():
    print(f"{name:20s}: {freq:9.1f} Hz, amplitude={amp:.5g}")